## 6 Итоговая оценка

**Входы**

- Выбранная модель, порог и test
- Метрики и правила оценивания из постановки задачи

**Выходы**

- Значения заданных метрик, графики и разбор ошибок

**Что делаем**

Рассчитаем на test метрики, заданные в этапе 1.2, и разберём ошибки. Модель и порог на этом шаге уже не меняем

In [ ]:
# dev-only: переходит в корень проекта; в сборку не входит
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


### 6.1 Метрики и ошибки

Рассчитываем метрики, заданные в этапе 1.2, и сравниваем с Dummy на том же test. В матрице ошибок строки соответствуют фактическому исходу, столбцы — прогнозу

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
from sklearn.calibration import calibration_curve

X_test = test.loc[:, FEATURES].copy()
y_test = test['loan_status'].astype(int).copy()
test_probability = selected_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype(int)
dummy_test_probability = trained_models['Dummy: prior'].predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([
    {'Решение': 'Dummy, порог 0.5', **classification_metrics(y_test, dummy_test_probability)},
    {'Решение': f'{selected_model_name}, порог validation', **classification_metrics(y_test, test_probability, selected_threshold)},
]).set_index('Решение')
test_confusion = confusion_matrix(y_test, test_prediction, labels=[0, 1])
tn, fp, fn, tp = test_confusion.ravel().tolist()
display(test_metrics.round(4))
display(pd.DataFrame(test_confusion, index=['Факт: без дефолта', 'Факт: дефолт'], columns=['Прогноз: без дефолта', 'Прогноз: дефолт']))


### 6.2 Разброс оценок

Используем bootstrap — выбор строк с повторами: 300 раз пересобираем test, сохраняя численность каждого класса. Показываем центральные 95% полученных значений метрик

Модель и порог фиксированы. Эти интервалы не учитывают новое обучение или изменение данных со временем

In [ ]:
# Стратифицированный bootstrap уже готовых прогнозов: модель/порог не меняются.
_bootstrap_rng = np.random.default_rng(42)
_bootstrap_y = y_test.to_numpy()
_negative_indices = np.flatnonzero(_bootstrap_y == 0)
_positive_indices = np.flatnonzero(_bootstrap_y == 1)
_bootstrap_metrics = []
for _ in range(300):
    _indices = np.concatenate([
        _bootstrap_rng.choice(_negative_indices, len(_negative_indices), replace=True),
        _bootstrap_rng.choice(_positive_indices, len(_positive_indices), replace=True),
    ])
    _bootstrap_metrics.append(classification_metrics(_bootstrap_y[_indices], test_probability[_indices], selected_threshold))
bootstrap_metrics = pd.DataFrame(_bootstrap_metrics)
metric_intervals = pd.DataFrame({
    'Оценка': test_metrics.iloc[1],
    'Нижняя 95%': bootstrap_metrics.quantile(.025),
    'Верхняя 95%': bootstrap_metrics.quantile(.975),
})
display(metric_intervals.round(4))


### 6.3 Диагностика прогнозов

ROC-кривая сопоставляет recall с долей ложных тревог $FP/(FP+TN)$, PR-кривая — precision с recall при разных порогах

График калибровки проверяет согласованность прогнозируемых вероятностей с частотами дефолта. По test ничего не донастраиваем

In [ ]:
# dev-only: подключает графики итоговой оценки
exec(compile(__import__('pathlib').Path('src/notebooks/_code/evaluation_plots.py').read_text(encoding='utf-8'), 'src/notebooks/_code/evaluation_plots.py', 'exec'), globals())


In [ ]:
display(Markdown(
    f'На test ({len(test):,} записей) AP = **{test_metrics.iloc[1]["AP"]:.4f}**, '
    f'ROC-AUC = **{test_metrics.iloc[1]["ROC-AUC"]:.4f}**. '
    f'При зафиксированном пороге обнаружено **{tp} из {tp + fn}** дефолтов; '
    f'пропущено **{fn}**, ложных тревог — **{fp}** из {tn + fp} кредитов без дефолта. '
    'После просмотра test решение не перенастраиваем'
))


#### Что ограничивает выводы

Мы проверили модель на одном случайном разбиении Kaggle-датасета. Не знаем время заявок, период наблюдения дефолта и то, встречается ли один человек несколько раз. Также неизвестно, насколько эти данные похожи на заявки конкретного банка. Редкие грейды представлены небольшим числом записей

Грейд и ставка должны быть известны до прогноза. Если нужна оценка до их назначения, состав признаков придётся пересмотреть и провести новый эксперимент

Для практического применения нужны проверки на более поздних и внешних данных, оценка стоимости ошибок и качества для разных групп заёмщиков. Полученные метрики сами по себе не обосновывают автоматический отказ в кредите